In [ ]:
import pandas as pd
import os
from pathlib import Path
pd.options.display.float_format = '{:,.2f}'.format

### ***This cell below is to read the parquet via jupyter***

df = pd.read_csv('../files/diario_ventas_new.csv', sep = ',', low_memory = False)

### ***This one is to read the parquet via vscode***

In [ ]:
base_dir = Path.cwd()
print(base_dir)
df = pd.read_parquet(f'{base_dir}/files/diario.parquet', engine = 'pyarrow')

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
df['ruta'] = df['ruta'].astype('Int64').astype(str)
variables_categoricas = ['centrocostos','unidadnegocios']
df[['cantidad','cantxunmedida']] = df[['cantidad','cantxunmedida']].astype('Int64')
df['fechafactura'] = pd.to_datetime(df['fechafactura'])

df[variables_categoricas] = df[variables_categoricas].astype(str)



In [ ]:
df.describe()

In [ ]:
group_dict = (
    df.dropna(subset=["grouptat"])
    .drop_duplicates(subset = ['referencia'])
    .set_index("referencia")['grouptat']
    .to_dict()
)
marca_dict = (
    df.dropna(subset=["tro_e_marca"])
    .drop_duplicates(subset = ['referencia'])
    .set_index("referencia")['tro_e_marca']
    .to_dict()
)
linea_dict = (
    df.dropna(subset=["lineatat"])
    .drop_duplicates(subset = ['referencia'])
    .set_index("referencia")['lineatat']
    .to_dict()
)
nombrereferencia_dict = (
    df.dropna(subset=["nombrereferencia"])
    .drop_duplicates(subset = ['referencia'])
    .set_index("referencia")['nombrereferencia']
    .to_dict()
)

In [ ]:
df.loc[df['porcentajemargen'].isnull(), 'porcentajemargen'] = 0
df.loc[df['porcentajedescuento'].isnull(), 'porcentajedescuento'] = 0
df.loc[df['ruta'].isnull(), 'ruta'] = 'NA'
df.loc[df['zona'].isnull(), 'zona'] = 'NA'
df.loc[df['ciclo'].isnull(), 'ciclo'] = 'NA'

df['grouptat'] = df['grouptat'].fillna(
    df['referencia'].map(group_dict)
)

df.loc[df['grouptat'].isnull(),'grouptat'] = 'NA'


df['tro_e_marca'] = df['tro_e_marca'].fillna(
    df['referencia'].map(marca_dict)
)
df.loc[df['tro_e_marca'].isnull(),'tro_e_marca'] = 'NA'

df['lineatat'] = df['lineatat'].fillna(
    df['referencia'].map(linea_dict)
)
df.loc[df['lineatat'].isnull(),'lineatat'] = 'NA'

df['nombrereferencia'] = df['nombrereferencia'].fillna(
    df['referencia'].map(nombrereferencia_dict)
)

In [ ]:
df.to_parquet(f'{base_dir}/files/sales.parquet')